# Homework 02 - Vector Search Answers

Notebook de respuestas para la Homework 2 del LLM Zoomcamp 2026.

Enunciado oficial: `homework.md`.

Setup usado:

```bash
uv add onnxruntime tokenizers numpy tqdm minsearch gitsource
uv add --dev huggingface-hub jupyter
uv run python download.py
```


## Final Answers

| Question | Answer |
|---|---|
| Q1 | `-0.02` |
| Q2 | `0.37` |
| Q3 | `02-vector-search/lessons/07-sqlitesearch-vector.md` |
| Q4 | `04-evaluation/lessons/05-search-metrics.md` |
| Q5 | `02-vector-search/lessons/08-pgvector.md` |
| Q6 | `01-agentic-rag/lessons/13-function-calling.md` |


## Imports and Constants

In [1]:
from pathlib import Path

import numpy as np
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import Index, VectorSearch
from tqdm.auto import tqdm

from embedder import Embedder

QUERY_Q1 = "How does approximate nearest neighbor search work?"
QUERY_Q4 = "What metric do we use to evaluate a search engine?"
QUERY_Q5 = "How do I store vectors in PostgreSQL?"
QUERY_Q6 = "How do I give the model access to tools?"

## Load Embedder and Data

In [2]:
embedder = Embedder("models/Xenova/all-MiniLM-L6-v2")

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]
chunks = chunk_documents(documents, size=2000, step=1000)

len(documents), len(chunks)

## Q1 - Embedding a Query

Embed the query and inspect the first value of the vector.

In [3]:
v = embedder.encode(QUERY_Q1)

print("v[0] =", float(v[0]))
print("Q1 answer: -0.02")

v[0] = -0.02058203437252893
Q1 answer: -0.02


## Q2 - Cosine Similarity

The vectors are normalized, so the dot product is cosine similarity.

In [4]:
target_doc = next(
    doc for doc in documents
    if doc["filename"] == "02-vector-search/lessons/07-sqlitesearch-vector.md"
)

target_vector = embedder.encode(target_doc["content"])
cosine = float(target_vector.dot(v))

print("cosine =", cosine)
print("Q2 answer: 0.37")

cosine = 0.36107026789538205
Q2 answer: 0.37


## Embed Chunks Once

The remaining questions reuse the chunk embeddings.

In [5]:
chunk_vectors = embedder.encode_batch([chunk["content"] for chunk in tqdm(chunks)])
X = np.vstack(chunk_vectors)

vector_search = VectorSearch(keyword_fields={"filename"})
vector_search.fit(X, chunks)

text_search = Index(text_fields=["content"], keyword_fields=["filename"])
text_search.fit(chunks)

## Q3 - Chunking and Search by Hand

In [6]:
scores = X.dot(v)
best_chunk = chunks[int(scores.argmax())]

print("best filename =", best_chunk["filename"])
print("best start =", best_chunk["start"])

best filename = 02-vector-search/lessons/07-sqlitesearch-vector.md
best start = 1000


## Q4 - Vector Search with minsearch

In [7]:
q4_vector = embedder.encode(QUERY_Q4)
q4_results = vector_search.search(q4_vector, num_results=5)

print([result["filename"] for result in q4_results])
print("Q4 answer:", q4_results[0]["filename"])

['04-evaluation/lessons/05-search-metrics.md', '04-evaluation/lessons/01-intro.md', '01-agentic-rag/lessons/05-search.md', '04-evaluation/lessons/01-intro.md', '04-evaluation/lessons/15-next-steps.md']
Q4 answer: 04-evaluation/lessons/05-search-metrics.md


## Q5 - Text Search vs Vector Search

In [8]:
q5_vector = embedder.encode(QUERY_Q5)
q5_vector_results = vector_search.search(q5_vector, num_results=5)
q5_text_results = text_search.search(QUERY_Q5, num_results=5)

q5_vector_files = [result["filename"] for result in q5_vector_results]
q5_text_files = [result["filename"] for result in q5_text_results]
q5_only_vector = sorted(set(q5_vector_files) - set(q5_text_files))

print("vector files:", q5_vector_files)
print("text files:", q5_text_files)
print("only vector:", q5_only_vector)

vector files: ['02-vector-search/lessons/08-pgvector.md', '02-vector-search/lessons/08-pgvector.md', '03-orchestration/lessons/05-rag.md', '02-vector-search/lessons/08-pgvector.md', '02-vector-search/lessons/08-pgvector.md']
text files: ['02-vector-search/lessons/02-embeddings.md', '03-orchestration/lessons/05-rag.md', '02-vector-search/lessons/01-intro.md', '03-orchestration/lessons/05-rag.md', '02-vector-search/lessons/01-intro.md']
only vector: ['02-vector-search/lessons/08-pgvector.md']


## Q6 - Hybrid Search with RRF

In [9]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}
    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]


q6_vector = embedder.encode(QUERY_Q6)
q6_vector_results = vector_search.search(q6_vector, num_results=5)
q6_text_results = text_search.search(QUERY_Q6, num_results=5)
q6_results = rrf([q6_vector_results, q6_text_results])

print("RRF files:", [result["filename"] for result in q6_results])
print("Q6 answer:", q6_results[0]["filename"])

RRF files: ['01-agentic-rag/lessons/13-function-calling.md', '01-agentic-rag/lessons/01-intro.md', '01-agentic-rag/lessons/14-agentic-loop.md', '04-evaluation/lessons/02-ground-truth.md', '01-agentic-rag/lessons/16-other-frameworks.md']
Q6 answer: 01-agentic-rag/lessons/13-function-calling.md


## Submission Dictionary

In [10]:
answers = {
    "Q1": "-0.02",
    "Q2": "0.37",
    "Q3": "02-vector-search/lessons/07-sqlitesearch-vector.md",
    "Q4": "04-evaluation/lessons/05-search-metrics.md",
    "Q5": "02-vector-search/lessons/08-pgvector.md",
    "Q6": "01-agentic-rag/lessons/13-function-calling.md",
}

answers

{'Q1': '-0.02',
 'Q2': '0.37',
 'Q3': '02-vector-search/lessons/07-sqlitesearch-vector.md',
 'Q4': '04-evaluation/lessons/05-search-metrics.md',
 'Q5': '02-vector-search/lessons/08-pgvector.md',
 'Q6': '01-agentic-rag/lessons/13-function-calling.md'}